# MA3632 — Workshop 2: Data Cleaning

This workshop accompanies Lecture 2. Part A covers missing value detection and
imputation. Part B covers outlier detection. Part C covers structural inconsistencies.
The `acquisition_report` function from Workshop 1 is reproduced at the top for
convenience; the new `cleaning_report` function introduced here plays an analogous
role after cleaning is applied.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(seed=42)

def acquisition_report(df, name="dataset"):
    """Concise pre-cleaning audit. Carried forward from Workshop 1."""
    n, p = df.shape
    missing = df.isnull().sum()
    missing_pct = (missing / n * 100).round(1)
    print(f"=== {name} ===")
    print(f"  Rows: {n}   Columns: {p}")
    print(f"  Dtypes: {dict(df.dtypes.value_counts())}")
    print()
    print("  Missing values:")
    if missing.sum() == 0:
        print("    None")
    else:
        for col in missing[missing > 0].sort_values(ascending=False).index:
            print(f"    {col:<22} {missing[col]:>4} ({missing_pct[col]}%)")
    print()
    print("  Duplicate rows:", df.duplicated().sum())
    print()
    print("  Numeric summary:")
    print(df.describe().round(2).to_string())
    print()

## Working dataset

We construct a synthetic patient dataset with known defects embedded deliberately:
missing values under a MAR mechanism, outliers, duplicates, a type error, and a
constraint violation. This gives us ground truth to verify each cleaning step against.

In [ ]:
n = 300

age        = rng.integers(18, 80, size=n).astype(float)
systolic   = 100 + 0.4 * age + rng.normal(0, 12, size=n)
diastolic  = 60  + 0.2 * age + rng.normal(0, 8,  size=n)
bmi        = rng.normal(26, 4, size=n)
region     = rng.choice(["North", "South", "East", "West"], size=n)
admitted   = rng.choice(["elective", "emergency"], size=n, p=[0.4, 0.6])

df = pd.DataFrame({
    "age":       age,
    "systolic":  systolic,
    "diastolic": diastolic,
    "bmi":       bmi,
    "region":    region,
    "admitted":  admitted,
})

# --- Embed defects ---

# 1. Missing values: systolic missing more often for elective admissions (MAR)
elective_idx = np.where(df["admitted"] == "elective")[0]
missing_idx  = rng.choice(elective_idx, size=30, replace=False)
df.loc[missing_idx, "systolic"] = np.nan

# 2. Missing at random: bmi missing for ~5% of records
bmi_missing = rng.choice(n, size=15, replace=False)
df.loc[bmi_missing, "bmi"] = np.nan

# 3. Outliers: a few extreme systolic values
df.loc[rng.choice(np.where(df["systolic"].notna())[0], 4, replace=False), "systolic"] += rng.choice([120, -80], 4)

# 4. Constraint violation: negative age
df.loc[rng.choice(n, 3, replace=False), "age"] = rng.choice([-5, -12, -3], 3)

# 5. Type error: region has an inconsistently cased entry
df.loc[rng.choice(n, 10, replace=False), "region"] = "north"

# 6. Duplicate rows
df = pd.concat([df, df.iloc[:5]], ignore_index=True)

acquisition_report(df, "Patient dataset (raw)")

---

## Part A — Missing values

### A1. Missingness audit

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

print("Missing counts and rates:")
print(pd.DataFrame({"count": missing, "pct": missing_pct}))

# Visualise missingness pattern
fig, ax = plt.subplots(figsize=(7, 3))
missing[missing > 0].plot(kind="barh", ax=ax, color="steelblue", edgecolor="none")
ax.set_xlabel("Missing value count")
ax.set_title("Missing values by column")
plt.tight_layout()
plt.show()

### A2. Identifying the mechanism

Before choosing an imputation strategy, we check whether missingness in `systolic`
is related to any observed variable. If it is, the mechanism is at most MAR.

In [ ]:
# Compare missingness rate in systolic across admission types
df["systolic_missing"] = df["systolic"].isnull().astype(int)

miss_by_admission = df.groupby("admitted")["systolic_missing"].agg(["sum", "mean"])
miss_by_admission.columns = ["n_missing", "rate"]
miss_by_admission["rate"] = miss_by_admission["rate"].round(3)
print(miss_by_admission)

# Also check by region (should be roughly equal if region is unrelated)
print()
print("Missingness rate by region:")
print(df.groupby("region")["systolic_missing"].mean().round(3))

The rate is substantially higher for elective admissions, confirming MAR: missingness
in `systolic` depends on the observed variable `admitted`, not on the blood pressure
value itself. This rules out MCAR and suggests that imputation should condition on
`admitted`.

### A3. Deletion

In [ ]:
# Complete-case analysis: discard any row with a missing value
df_cc = df.dropna().copy()
print(f"Original: {len(df)} rows")
print(f"After listwise deletion: {len(df_cc)} rows  ({len(df_cc)/len(df)*100:.1f}% retained)")

# How does the retained sample compare on admission type?
print("\nAdmission type distribution — original:")
print(df["admitted"].value_counts(normalize=True).round(3))
print("\nAdmission type distribution — complete cases:")
print(df_cc["admitted"].value_counts(normalize=True).round(3))

Complete-case analysis over-represents emergency admissions because elective patients
are more likely to have a missing `systolic`. Any model fitted on the complete cases
will be biased on this variable. This is the practical consequence of analysing
incomplete data under MAR without imputation.

### A4. Mean and median imputation

In [ ]:
df_imp = df.copy()

# Mean imputation for systolic (computed on observed values only)
systolic_mean = df_imp["systolic"].mean()
systolic_med  = df_imp["systolic"].median()

df_mean = df_imp.copy()
df_mean["systolic"] = df_mean["systolic"].fillna(systolic_mean)

df_med = df_imp.copy()
df_med["systolic"] = df_med["systolic"].fillna(systolic_med)

print(f"Observed mean:   {systolic_mean:.2f}")
print(f"Observed median: {systolic_med:.2f}")

# Compare variance before and after
print(f"\nVariance (observed only): {df_imp['systolic'].var(ddof=1):.2f}")
print(f"Variance (mean imputed):  {df_mean['systolic'].var(ddof=1):.2f}")
print(f"Variance (median imputed):{df_med['systolic'].var(ddof=1):.2f}")

In [ ]:
# Illustrate variance attenuation
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, data, title in zip(
    axes,
    [df_imp["systolic"].dropna(), df_mean["systolic"], df_med["systolic"]],
    ["Observed only", "Mean imputed", "Median imputed"]
):
    ax.hist(data, bins=40, color="steelblue", edgecolor="none", alpha=0.85)
    ax.axvline(data.mean(), color="firebrick", linewidth=1.2, linestyle="--", label="mean")
    ax.set_title(title)
    ax.set_xlabel("Systolic BP")
    ax.legend(fontsize=8)

fig.suptitle("Effect of single imputation on the distribution of systolic BP", fontsize=10)
plt.tight_layout()
plt.show()

### A5. Regression imputation

In [ ]:
from sklearn.linear_model import LinearRegression

# Fit on complete cases using admitted (encoded) and age as predictors
df_reg = df.copy()
df_reg["elective"] = (df_reg["admitted"] == "elective").astype(int)

complete = df_reg[df_reg["systolic"].notna()]
missing_rows = df_reg[df_reg["systolic"].isna()]

predictors = ["age", "elective"]
model = LinearRegression()
model.fit(complete[predictors], complete["systolic"])

df_reg.loc[df_reg["systolic"].isna(), "systolic"] = model.predict(missing_rows[predictors])

print("Regression imputation fitted.")
print(f"Coefficients — age: {model.coef_[0]:.3f},  elective: {model.coef_[1]:.3f}")
print(f"Intercept: {model.intercept_:.3f}")
print()
print(f"Variance (regression imputed): {df_reg['systolic'].var(ddof=1):.2f}")
print(f"Variance (mean imputed):       {df_mean['systolic'].var(ddof=1):.2f}")

Regression imputation produces imputed values that vary with `age` and `admitted`,
preserving more of the original correlation structure than mean imputation does.
The variance is still slightly deflated relative to the observed-only sample because
imputed values are point predictions with no residual noise added.

**Exercise A.** The variable `bmi` also has missing values. Using `acquisition_report`
on the raw dataset as a guide, argue whether MCAR or MAR is more plausible for the
`bmi` missingness. Then apply mean imputation to `bmi` in the regression-imputed
dataset `df_reg`, and verify numerically that the post-imputation variance is reduced
by the factor derived in Lecture 2.

---

## Part B — Outlier detection

We work on `df_reg` (missing values already handled) so that outlier detection
operates on a complete dataset.

### B1. Univariate detection: Tukey fences

In [ ]:
def tukey_outliers(series, k=1.5):
    """Return boolean mask of outliers using the Tukey fence criterion."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - k * iqr) | (series > q3 + k * iqr)

numeric_cols = ["age", "systolic", "diastolic", "bmi"]

for col in numeric_cols:
    mask = tukey_outliers(df_reg[col].dropna())
    n_out = mask.sum()
    print(f"{col:<12}  IQR fence outliers: {n_out}  ({n_out/len(mask)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, col in zip(axes, numeric_cols):
    data = df_reg[col].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor="steelblue", alpha=0.6))
    ax.axhline(upper, color="firebrick", linestyle="--", linewidth=0.9, label="fence")
    ax.axhline(lower, color="firebrick", linestyle="--", linewidth=0.9)
    ax.set_title(col)
    ax.legend(fontsize=7)

plt.suptitle("Boxplots with Tukey fences", fontsize=10)
plt.tight_layout()
plt.show()

### B2. Z-scores and the masking problem

In [ ]:
def zscore_outliers(series, threshold=3.0):
    z = (series - series.mean()) / series.std(ddof=1)
    return z.abs() > threshold, z

mask_z, z = zscore_outliers(df_reg["systolic"].dropna())
print(f"Systolic z-score outliers (|z| > 3): {mask_z.sum()}")

# Now check: does inserting an extreme value mask itself?
s_with_extreme = df_reg["systolic"].dropna().copy()
s_with_extreme.iloc[0] = 400   # extreme artificial value

mask_z2, z2 = zscore_outliers(s_with_extreme)
print(f"\nAfter inserting value of 400:")
print(f"  z-score of the inserted value: {z2.iloc[0]:.2f}")
print(f"  Total z-score outliers: {mask_z2.sum()}")
print(f"  Mean shifted to: {s_with_extreme.mean():.2f}  (was {df_reg['systolic'].mean():.2f})")

In [ ]:
# Robust alternative: median and MAD
def mad_outliers(series, threshold=3.5):
    med = series.median()
    mad = (series - med).abs().median()
    modified_z = 0.6745 * (series - med) / mad  # scale factor for consistency with normal
    return modified_z.abs() > threshold, modified_z

mask_mad, mz = mad_outliers(s_with_extreme)
print(f"MAD-based outliers after inserting 400: {mask_mad.sum()}")
print(f"Modified z-score of inserted value: {mz.iloc[0]:.2f}")

The standard z-score inflates the mean and standard deviation when an extreme value is
present, compressing all other z-scores toward zero and potentially hiding the outlier.
The MAD-based statistic uses the median and is resistant to this effect.

### B3. Multivariate detection: Mahalanobis distance

In [ ]:
from scipy.spatial.distance import mahalanobis
from scipy.stats import chi2

data_mv = df_reg[["systolic", "diastolic"]].dropna().values

mu    = data_mv.mean(axis=0)
Sigma = np.cov(data_mv, rowvar=False)
Sigma_inv = np.linalg.inv(Sigma)

D2 = np.array([mahalanobis(x, mu, Sigma_inv)**2 for x in data_mv])

# Chi-squared threshold at 97.5% with p=2 degrees of freedom
threshold = chi2.ppf(0.975, df=2)
outlier_mask = D2 > threshold

print(f"Mahalanobis outliers (chi2 threshold at 97.5%, df=2): {outlier_mask.sum()}")
print(f"Threshold D^2 = {threshold:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(data_mv[~outlier_mask, 0], data_mv[~outlier_mask, 1],
           alpha=0.4, s=18, color="steelblue", label="normal")
ax.scatter(data_mv[outlier_mask, 0], data_mv[outlier_mask, 1],
           alpha=0.9, s=40, color="firebrick", marker="x", linewidths=1.5,
           label="Mahalanobis outlier")
ax.set_xlabel("Systolic BP")
ax.set_ylabel("Diastolic BP")
ax.set_title("Mahalanobis distance outliers: systolic vs diastolic")
ax.legend()
plt.tight_layout()
plt.show()

**Exercise B.** Apply the Tukey fence criterion and the Mahalanobis criterion to
`age` alone and to the pair `(age, bmi)` respectively. Are the same observations
flagged by both methods? If not, explain geometrically why they can disagree.

---

## Part C — Structural inconsistencies

### C1. Duplicate records

In [ ]:
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")

df_clean = df.drop_duplicates().copy()
print(f"Rows after deduplication: {len(df_clean)}  (removed {len(df) - len(df_clean)})")

### C2. Type and format errors

In [ ]:
print("Unique values in 'region':")
print(sorted(df_clean["region"].unique()))

In [ ]:
# Normalise region to title case
df_clean["region"] = df_clean["region"].str.strip().str.title()

print("After normalisation:")
print(sorted(df_clean["region"].unique()))
print("\nValue counts:")
print(df_clean["region"].value_counts())

### C3. Constraint violations

In [ ]:
# Age must be in [0, 120]
invalid_age = df_clean[df_clean["age"] < 0]
print(f"Records with negative age: {len(invalid_age)}")
print(invalid_age[["age", "systolic", "admitted"]])

In [ ]:
# Recode invalid ages as missing; they cannot be corrected without the source data
df_clean.loc[df_clean["age"] < 0, "age"] = np.nan
print(f"\nMissing age values after recoding: {df_clean['age'].isna().sum()}")

---

## Cleaning report

A simple function to compare the dataset before and after cleaning.

In [ ]:
def cleaning_report(df_before, df_after, name="dataset"):
    """Compare row count, missing values, and duplicates before and after cleaning."""
    print(f"=== Cleaning report: {name} ===")
    print(f"  Rows:      {len(df_before)} -> {len(df_after)}")
    print(f"  Duplicates removed: {df_before.duplicated().sum()}")
    print()

    miss_before = df_before.isnull().sum()
    miss_after  = df_after.isnull().sum()

    print("  Missing values (before -> after):")
    for col in df_before.columns:
        b, a = miss_before[col], miss_after.get(col, 0)
        if b > 0 or a > 0:
            print(f"    {col:<22} {b:>4} -> {a:>4}")
    print()

cleaning_report(df, df_clean, "Patient dataset")

---

## Take-home exercises

**Exercise 1.** The `systolic` missingness was MAR, depending on `admitted`. Explain
why imputing with the overall observed mean is biased in this case, and show
numerically that the mean of the regression-imputed values for the originally-missing
rows differs from `systolic_mean`.

**Exercise 2.** Using the formula derived in Lecture 2, compute the expected
post-imputation variance of `systolic` after mean imputation as a function of the
observed variance and the number of missing values. Compare this to the value
you computed in Part A and verify that they agree.

**Exercise 3.** Apply the full cleaning workflow from Lecture 2 (audit, structural
repair, deduplication, outlier investigation, imputation, re-audit) to the Titanic
dataset from Workshop 1. Document each decision you make and the reason for it.

**Exercise 4.** (Written, no code.) Suppose you are given a dataset where 40% of
values in one column are missing. A colleague suggests discarding the column entirely.
Under what conditions is this reasonable, and under what conditions might it discard
useful information? How does the missingness mechanism bear on this decision?

---